# Laboratorio 1: Generación de Música
## Modelo Autoregresivo con Transformer (Frame-Level GPT)

Entrenamos un Transformer decoder compacto (~2M params) que predice autoregressivamente el siguiente frame de piano roll (vector binario de 88 dimensiones) dado un contexto de frames anteriores.

- **Arquitectura**: Transformer Decoder, 4 capas, 4 cabezas, d_model=256
- **Datos**: 10,604 secuencias musicales en formato piano roll [T, 88]

## 1. Imports y configuración del dispositivo

In [ ]:
import time
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from tqdm.auto import tqdm

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Ruta base en Drive
DRIVE_BASE = "/content/drive/MyDrive/musicbox"

# Copiar datos a disco local (mucho más rápido que leer de Drive en cada batch)
LOCAL_DATA = "/content/musicbox_local"
import os
os.makedirs(LOCAL_DATA, exist_ok=True)
if not os.path.exists(f"{LOCAL_DATA}/train.npz"):
    print("Copiando train.npz a disco local...")
    shutil.copy(f"{DRIVE_BASE}/train.npz", f"{LOCAL_DATA}/train.npz")
    print("Listo.")

shutil.copy(f"{DRIVE_BASE}/audio_play.py", f"{LOCAL_DATA}/audio_play.py")

import sys
sys.path.insert(0, LOCAL_DATA)
import audio_play

# Dispositivo
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Usando dispositivo: {device}")
print(f"PyTorch version: {torch.__version__}")

## 2. Carga y exploración de datos

In [ ]:
NPZ_PATH = f"{LOCAL_DATA}/train.npz"

data = audio_play.load_musicbox_dataset(NPZ_PATH)
defaults = audio_play.get_dataset_defaults(data)
print("Dataset defaults:", defaults)

rolls_flat = data["rolls_flat"]  # (total_frames, 88) uint8
offsets = data["offsets"]         # (num_seqs + 1,)
num_sequences = len(offsets) - 1

print(f"Número de secuencias: {num_sequences}")
print(f"Total de frames: {len(rolls_flat):,}")
print(f"Shape de rolls_flat: {rolls_flat.shape}")
print(f"Densidad media de notas: {rolls_flat.astype(float).mean():.4f}")

# Distribución de longitudes
lengths = [int(offsets[i+1]) - int(offsets[i]) for i in range(num_sequences)]
print(f"Longitud min/max/media: {min(lengths)}/{max(lengths)}/{np.mean(lengths):.0f}")

In [ ]:
# Visualizar una secuencia de ejemplo
seq_idx = 0
seq = audio_play.get_sequence_from_dataset(data, seq_idx)
print(f"Secuencia {seq_idx}: shape={seq.shape}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(seq[:512].T, aspect="auto", origin="lower", cmap="Greys", interpolation="none")
ax.set_xlabel("Paso temporal")
ax.set_ylabel("Nota MIDI (offset)")
ax.set_title(f"Piano Roll - Secuencia {seq_idx} (primeros 512 pasos)")
plt.tight_layout()
plt.show()

In [ ]:
# Escuchar la secuencia original
audio_orig, sr = audio_play.synthesize_musicbox_roll(
    seq[:512],
    step_sec=defaults["step_sec"],
    note_min=defaults["note_min"],
    representation=defaults["representation"],
)
print("Audio original (primeros 512 pasos):")
display(Audio(audio_orig, rate=sr))

## 3. Dataset y DataLoader

In [ ]:
# Hiperparámetros
SEQ_LEN = 256       # Longitud de contexto (en frames)
BATCH_SIZE = 64
NUM_WORKERS = 2     # Workers para cargar datos en paralelo (CUDA soporta esto bien)


class MusicRollDataset(Dataset):
    """Dataset que extrae ventanas de longitud fija del piano roll."""

    def __init__(self, rolls_flat, offsets, seq_len):
        # Copiar a numpy contiguo en RAM para acceso rápido
        self.rolls_flat = np.ascontiguousarray(rolls_flat)
        self.seq_len = seq_len
        # Pre-calcular todos los posibles puntos de inicio (ventanas válidas)
        self.windows = []
        num_seqs = len(offsets) - 1
        for i in range(num_seqs):
            start = int(offsets[i])
            end = int(offsets[i + 1])
            length = end - start
            if length > seq_len:
                # Ventanas con stride de seq_len//2 para overlap
                stride = max(1, seq_len // 2)
                for j in range(start, end - seq_len, stride):
                    self.windows.append(j)

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        start = self.windows[idx]
        # seq_len + 1 frames: primeros seq_len son input, últimos seq_len son target
        chunk = self.rolls_flat[start : start + self.seq_len + 1].astype(np.float32)
        x = torch.from_numpy(chunk[:-1])   # (seq_len, 88)
        y = torch.from_numpy(chunk[1:])    # (seq_len, 88)
        return x, y


dataset = MusicRollDataset(rolls_flat, offsets, SEQ_LEN)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True, pin_memory=True,
    persistent_workers=True,
)

print(f"Ventanas de entrenamiento: {len(dataset):,}")
print(f"Batches por época: {len(dataloader):,}")

# Verificar un batch
x_sample, y_sample = next(iter(dataloader))
print(f"Batch x shape: {x_sample.shape}")
print(f"Batch y shape: {y_sample.shape}")
print(f"Densidad media en batch: {x_sample.mean():.4f}")

## 4. Modelo: Music Transformer (Frame-Level GPT)

In [ ]:
class MusicTransformer(nn.Module):
    def __init__(
        self,
        num_notes=88,
        d_model=256,
        nhead=4,
        num_layers=4,
        d_ff=512,
        max_seq_len=256,
        dropout=0.1,
    ):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # Proyección de frame binario a embedding
        self.frame_embed = nn.Linear(num_notes, d_model)

        # Positional encoding aprendido
        self.pos_embed = nn.Embedding(max_seq_len, d_model)

        # Transformer decoder layers
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,  # Pre-LayerNorm (más estable)
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)

        # Capa de salida
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_notes)

        # Máscara causal (se registra como buffer)
        mask = nn.Transformer.generate_square_subsequent_mask(max_seq_len)
        self.register_buffer("causal_mask", mask)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        """x: (batch, seq_len, 88) -> logits: (batch, seq_len, 88)"""
        B, T, _ = x.shape

        # Embedding
        h = self.frame_embed(x)  # (B, T, d_model)
        positions = torch.arange(T, device=x.device)
        h = h + self.pos_embed(positions)  # broadcast over batch

        # Máscara causal
        mask = self.causal_mask[:T, :T]

        # Transformer con máscara causal (decoder-only = encoder con máscara causal)
        h = self.transformer(h, mask=mask, is_causal=True)

        # Proyección de salida
        h = self.ln_f(h)
        logits = self.head(h)  # (B, T, 88)
        return logits


# Crear modelo
model = MusicTransformer(
    num_notes=88,
    d_model=256,
    nhead=4,
    num_layers=4,
    d_ff=512,
    max_seq_len=SEQ_LEN,
    dropout=0.1,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros del modelo: {num_params:,}")
print(model)

## 5. Entrenamiento

In [ ]:
# Configuración de entrenamiento
MAX_EPOCHS = 100
LR = 3e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0
PATIENCE = 10          # Early stopping con más paciencia para entrenar profundo
WARMUP_EPOCHS = 5      # Warmup lineal antes de cosine decay

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Scheduler: warmup lineal + cosine decay
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS  # warmup lineal
    progress = (epoch - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))  # cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# BCEWithLogitsLoss con pos_weight moderado
pos_weight = torch.tensor([3.0] * 88, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Entrenamiento
train_losses = []
best_loss = float("inf")
epochs_without_improvement = 0
start_time = time.time()

print(f"Iniciando entrenamiento: {MAX_EPOCHS} épocas máx, early stopping paciencia={PATIENCE}")
print(f"Warmup: {WARMUP_EPOCHS} épocas")
print(f"Batches por época: {len(dataloader)}")
print("-" * 60)

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    for x, y in dataloader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    elapsed = (time.time() - start_time) / 60

    scheduler.step()
    avg_loss = epoch_loss / max(num_batches, 1)
    train_losses.append(avg_loss)
    lr_now = optimizer.param_groups[0]["lr"]

    # Early stopping check
    if avg_loss < best_loss:
        best_loss = avg_loss
        epochs_without_improvement = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1

    # Print solo 1 línea por época (sin tqdm interno = mucho más rápido en Colab)
    print(f"Época {epoch+1:3d}/{MAX_EPOCHS} | Loss: {avg_loss:.5f} | Best: {best_loss:.5f} | LR: {lr_now:.2e} | Pat: {epochs_without_improvement}/{PATIENCE} | {elapsed:.1f}m")

    if epochs_without_improvement >= PATIENCE:
        print(f"\n⏹ Early stopping en época {epoch+1} (sin mejora por {PATIENCE} épocas)")
        break

# Restaurar mejor modelo
model.load_state_dict(best_state)
print(f"\nModelo restaurado al mejor checkpoint (loss={best_loss:.5f})")

total_time = (time.time() - start_time) / 60
print(f"Entrenamiento completado en {total_time:.1f} minutos")
print(f"Épocas completadas: {len(train_losses)}")

# Guardar checkpoint en Drive
os.makedirs(f"{DRIVE_BASE}/checkpoints", exist_ok=True)
checkpoint_path = f"{DRIVE_BASE}/checkpoints/model_checkpoint.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "train_losses": train_losses,
    "epoch": len(train_losses),
    "best_loss": best_loss,
}, checkpoint_path)
print(f"Checkpoint guardado en {checkpoint_path}")

In [ ]:
# Curva de pérdida
plt.figure(figsize=(10, 4))
plt.plot(train_losses, marker="o", markersize=3)
plt.xlabel("Época")
plt.ylabel("BCE Loss")
plt.title("Curva de Entrenamiento")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Generación de música

In [ ]:
@torch.no_grad()
def generate(
    model,
    seed_frames,
    num_steps=512,
    temperature=1.0,
    top_p=0.9,
    max_notes=8,
    repetition_penalty=0.85,
    repetition_window=16,
    device="cuda",
):
    """
    Genera frames de piano roll autoregressivamente con sampling mejorado.

    En vez de aplicar temperatura a sigmoids independientes (lo cual explota/colapsa),
    usamos un enfoque tipo "nucleus sampling adaptado":
    1. Convertir los 88 logits en una distribución categorical via softmax
    2. Samplear cuántas notas activar (basado en la distribución aprendida)
    3. Elegir cuáles notas activar usando top-p sobre esa distribución
    4. Aplicar penalización por repetición

    Args:
        temperature: controla diversidad en la selección de notas (0.5-1.5 rango útil)
        top_p: nucleus sampling - solo considerar notas cuya prob acumulada < top_p
        max_notes: máximo de notas por frame (safety cap)
        repetition_penalty: factor para penalizar notas que ya aparecieron recientemente
        repetition_window: cuántos frames atrás mirar para repetición
    """
    model.eval()
    max_ctx = model.max_seq_len

    if isinstance(seed_frames, np.ndarray):
        seed_frames = torch.from_numpy(seed_frames.astype(np.float32))
    
    generated = seed_frames.clone().to(device)  # (current_len, 88)

    for step in range(num_steps):
        # Tomar los últimos max_ctx frames como contexto
        ctx = generated[-max_ctx:].unsqueeze(0)  # (1, ctx_len, 88)
        logits = model(ctx)  # (1, ctx_len, 88)
        next_logits = logits[0, -1, :]  # (88,)

        # --- Penalización por repetición ---
        # Si una nota ha estado activa muchas veces en los últimos N frames, reducir su logit
        if repetition_penalty < 1.0 and generated.shape[0] >= repetition_window:
            recent = generated[-repetition_window:]  # (window, 88)
            note_frequency = recent.mean(dim=0)  # (88,) qué tan frecuente estuvo cada nota
            # Penalizar proporcionalmente a la frecuencia reciente
            penalty = 1.0 - (1.0 - repetition_penalty) * note_frequency
            next_logits = next_logits * penalty

        # --- Determinar cuántas notas activar ---
        # Usar sigmoid para estimar la densidad esperada, luego samplear un conteo
        probs_raw = torch.sigmoid(next_logits)
        expected_notes = probs_raw.sum().item()
        # Samplear N ~ Poisson(expected_notes), clamp a [0, max_notes]
        num_notes = min(max_notes, max(0, int(torch.poisson(torch.tensor(expected_notes)).item())))

        if num_notes == 0:
            # Frame silencioso
            next_frame = torch.zeros(88, device=device)
        else:
            # --- Nucleus (top-p) sampling sobre logits con softmax ---
            # Tratar los 88 logits como una distribución categorical para elegir CUÁLES notas
            scaled_logits = next_logits / temperature
            probs_softmax = F.softmax(scaled_logits, dim=0)  # (88,)

            # Top-p filtering
            sorted_probs, sorted_indices = torch.sort(probs_softmax, descending=True)
            cumulative_probs = torch.cumsum(sorted_probs, dim=0)
            # Encontrar el corte donde la masa acumulada supera top_p
            cutoff_idx = torch.searchsorted(cumulative_probs, top_p).item() + 1
            cutoff_idx = max(cutoff_idx, num_notes)  # al menos num_notes candidatos
            cutoff_idx = min(cutoff_idx, 88)

            # Zero out notas fuera del nucleus
            candidate_indices = sorted_indices[:cutoff_idx]
            candidate_probs = sorted_probs[:cutoff_idx]
            candidate_probs = candidate_probs / candidate_probs.sum()  # renormalizar

            # Samplear num_notes notas sin reemplazo
            num_to_sample = min(num_notes, len(candidate_indices))
            chosen = torch.multinomial(candidate_probs, num_to_sample, replacement=False)
            chosen_notes = candidate_indices[chosen]

            next_frame = torch.zeros(88, device=device)
            next_frame[chosen_notes] = 1.0

        generated = torch.cat([generated, next_frame.unsqueeze(0)], dim=0)

    # Retornar solo la parte generada (sin seed)
    result = generated[len(seed_frames):].cpu().numpy().astype(np.uint8)
    return result


print("Función de generación definida.")

In [ ]:
# Generar música a partir de un seed real
seed_seq_idx = 0
seed_seq = audio_play.get_sequence_from_dataset(data, seed_seq_idx)
seed_len = 64  # Más contexto inicial para mejor arranque
seed = seed_seq[:seed_len]

print(f"Semilla: secuencia {seed_seq_idx}, primeros {seed_len} frames")
print(f"Notas activas en semilla: {seed.sum()}")
print(f"Densidad semilla: {seed.astype(float).mean():.4f}")

# Generar con diferentes temperaturas
temperatures = [0.7, 1.0, 1.3]
generated_rolls = {}

for temp in temperatures:
    print(f"\nGenerando con temperatura={temp}...")
    gen = generate(
        model, seed, num_steps=512,
        temperature=temp, top_p=0.9, max_notes=6,
        repetition_penalty=0.85, repetition_window=16,
        device=device
    )
    generated_rolls[temp] = gen
    print(f"  Frames generados: {gen.shape}")
    print(f"  Densidad de notas: {gen.astype(float).mean():.4f}")
    print(f"  Notas totales: {gen.sum()}")
    print(f"  Frames vacíos: {(gen.sum(axis=1) == 0).sum()}/{len(gen)}")

In [ ]:
# Sintetizar y reproducir audio generado
for temp, roll in generated_rolls.items():
    print(f"\n--- Temperatura {temp} ---")

    # Combinar semilla + generado para continuidad
    full_roll = np.concatenate([seed, roll], axis=0)

    audio_gen, sr = audio_play.synthesize_musicbox_roll(
        full_roll,
        step_sec=defaults["step_sec"],
        note_min=defaults["note_min"],
        representation=defaults["representation"],
    )
    display(Audio(audio_gen, rate=sr))

    # Guardar WAV en Drive
    out_path = f"{DRIVE_BASE}/generated_temp{temp}.wav"
    audio_play.save_wav(out_path, audio_gen, sr)
    print(f"  Guardado en {out_path}")

## 7. Visualización: Real vs Generado

In [ ]:
# Comparación visual: secuencia real vs generada
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Real (continuación verdadera después de la semilla)
real_continuation = seed_seq[seed_len:seed_len + 512]
axes[0].imshow(real_continuation.T, aspect="auto", origin="lower", cmap="Greys", interpolation="none")
axes[0].set_ylabel("Nota")
axes[0].set_title("Continuación REAL (512 pasos después de semilla)")

# Generado (temperatura 1.0)
gen_roll = generated_rolls[1.0]
axes[1].imshow(gen_roll[:512].T, aspect="auto", origin="lower", cmap="Greys", interpolation="none")
axes[1].set_ylabel("Nota")
axes[1].set_xlabel("Paso temporal")
axes[1].set_title("Continuación GENERADA (T=1.0, 512 pasos)")

plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas comparativas
print("Estadísticas comparativas (512 frames):")
print(f"{'':>25s} {'Real':>10s} {'Gen T=0.7':>10s} {'Gen T=1.0':>10s} {'Gen T=1.3':>10s}")
print("-" * 65)

real_c = real_continuation[:512].astype(float)
stats = {
    "Densidad (mean)": [real_c.mean()],
    "Notas/frame (mean)": [real_c.sum(axis=1).mean()],
    "Total notas": [real_c.sum()],
    "Frames vacíos (%)": [(real_c.sum(axis=1) == 0).mean() * 100],
}

for temp in temperatures:
    g = generated_rolls[temp][:512].astype(float)
    stats["Densidad (mean)"].append(g.mean())
    stats["Notas/frame (mean)"].append(g.sum(axis=1).mean())
    stats["Total notas"].append(g.sum())
    stats["Frames vacíos (%)"].append((g.sum(axis=1) == 0).mean() * 100)

for name, vals in stats.items():
    row = f"{name:>25s}"
    for v in vals:
        row += f" {v:10.2f}"
    print(row)

print("\nAudio de la continuación real para comparación:")
audio_real, sr = audio_play.synthesize_musicbox_roll(
    real_continuation[:512],
    step_sec=defaults["step_sec"],
    note_min=defaults["note_min"],
    representation=defaults["representation"],
)
display(Audio(audio_real, rate=sr))

## 8. Generación libre (sin semilla real)

In [ ]:
# Generar desde un frame vacío (generación completamente libre)
empty_seed = np.zeros((1, 88), dtype=np.float32)

print("Generando música desde cero (semilla vacía)...")
gen_free = generate(
    model, empty_seed, num_steps=1024,
    temperature=1.0, top_p=0.9, max_notes=5,
    repetition_penalty=0.8, repetition_window=24,
    device=device
)

print(f"Frames generados: {gen_free.shape}")
print(f"Densidad: {gen_free.astype(float).mean():.4f}")

# Visualizar
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(gen_free.T, aspect="auto", origin="lower", cmap="Greys", interpolation="none")
ax.set_xlabel("Paso temporal")
ax.set_ylabel("Nota")
ax.set_title("Generación libre (1024 pasos, T=1.0, top_p=0.9)")
plt.tight_layout()
plt.show()

# Sintetizar
audio_free, sr = audio_play.synthesize_musicbox_roll(
    gen_free,
    step_sec=defaults["step_sec"],
    note_min=defaults["note_min"],
    representation=defaults["representation"],
)
print("Audio generado libremente:")
display(Audio(audio_free, rate=sr))

audio_play.save_wav(f"{DRIVE_BASE}/generated_free.wav", audio_free, sr)
print(f"Guardado en {DRIVE_BASE}/generated_free.wav")